# Case 13 -- Cape May, New Jersey Cross-Section (SHARP example 2)

Steady two-fluid, three-aquifer coastal cross-section from Essaid (1990, USGS WRIR
90-4130, figures 18-19 and 23-24, tables 4, 9 and 10), after Hill (1988). This is a
code-to-code comparison with SHARP using its published input deck
(`ref/sharp/SHARP1.1/DATA/INPUT.T2`) and solution (`RESULTS.T2`). An upper water-table
aquifer over two confined aquifers separated by confining beds, 88,000 ft long.
Freshwater enters by leakage through the top onshore under a parabolic water table;
offshore the overlying water is the sea. Hill (1988) also ran this problem with SUTRA, so
figure 19A shows the sharp interface with chloride contours from a variable-density
model.

SHARP numbers aquifers from the bottom and MODFLOW numbers layers from the top. Results
below use SHARP aquifer and column numbers, with `IAQ1`/`IAQ2`/`IAQ3` the corresponding
MF6 layer indices. SHARP columns 1 and 28 are inactive, so MF6 column $j$ is SHARP
column $j + 2$.

In [ ]:
import json
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np

# path to mf6 executables with swi support:
#   https://github.com/christianlangevin/modflow6-nightly-build/actions/workflows/nightly-build-swi.yml

# Put the name of the mf6 executable into mf6exe.txt,
# which is not under version control.
with open(pl.Path("./mf6exe.txt"), "r") as f:
    mf6exe = f.readline().strip()
print(f"using executable: {mf6exe}")

sim_ws = pl.Path("./temp/case13")
SHARP_DATA = pl.Path("../ref/sharp/SHARP1.1/DATA")

## Reading SHARP's files

`INPUT.T2` and `RESULTS.T2` are fixed-format listings: a control line, then a 3 x 28
array in records of 10, 10 and 8 values per row, blank-filled, so they are read record
by record. Only row 2 is active.

In [ ]:
def sharp_row2(path, ctrl_line0):
    """Row 2 of a 3 x 28 SHARP array whose control line is at 0-based ctrl_line0."""
    lines = path.read_text().splitlines()
    i, rows = ctrl_line0 + 1, []
    for _ in range(3):
        v = []
        for _ in range(3):
            v += [float(t) for t in lines[i].split()]
            i += 1
        v += [0.0] * (28 - len(v))          # short records are blank-filled
        rows.append(v[:28])
    return np.array(rows[1])


def sharp_arrays(path, pattern):
    """All arrays in a SHARP file whose control line matches a regex."""
    import re
    lines = path.read_text().splitlines()
    out = {}
    for n, line in enumerate(lines):
        m = re.search(pattern, line)
        if m:
            out["".join(m.groups())] = sharp_row2(path, n)
    return out


INPUT = SHARP_DATA / "INPUT.T2"
RESULTS = SHARP_DATA / "RESULTS.T2"
sharp = sharp_arrays(RESULTS, r"(ZINT|PHIF)\s+LAYER\s+(\d)")
print("SHARP solution arrays read:", ", ".join(sorted(sharp)))
print(f"ZINT1 (bottom aquifer), SHARP cols 2-11:")
print("   " + " ".join(f"{sharp['ZINT1'][c-1]:9.3f}" for c in range(2, 12)))

## Parameters

From `INPUT.T2`, cross-checked against figure 23. $\gamma_f$ = 62.4100 and
$\gamma_s$ = 63.97025 lb/$\mathrm{ft}^3$ give $\alpha_f$ = 40. The anisotropy of 100 in table 4
is a SUTRA parameter; SHARP puts all vertical resistance in the confining beds, so the
aquifers are left vertically well connected (`k33 = k`).

In [ ]:
gamf, gams = 62.4100, 63.97025             # INPUT.T2 line 3
alphaf = gamf / (gams - gamf)              # 40.0
alphas = gams / (gams - gamf)              # 41.0

# grid: SHARP columns 2-27 are active; DELX and DELY from the end of INPUT.T2
delr = np.array([4000.0] * 5 + [2000.0] * 8 + [4000.0] * 13)
ncol = delr.size                           # 26
xcen = np.cumsum(delr) - 0.5 * delr
XSHORE = 5 * 4000.0                        # shoreline at the col 6/7 boundary
x = xcen - XSHORE                          # ft from shoreline, as in fig. 19
delc = 100.0                               # DELY

# layering (fig. 23 and THCK/ZBOT in INPUT.T2), MF6 order = top down
top = -10.0
botm = [-30.0, -50.0, -90.0, -110.0, -190.0]
IAQ3, ICONF32, IAQ2, ICONF21, IAQ1 = 0, 1, 2, 3, 4
nlay = 5

k_aq3, k_aq2, k_aq1 = 4.64e-4, 9.0e-4, 1.7e-3        # FKX x 29.0E-07
LEAK_TOP, LEAK_32, LEAK_21 = 4.64e-7, 5.00e-10, 3.73e-8   # AQL multipliers, 1/s

kh = np.zeros((nlay, 1, ncol))
kh[IAQ3] = k_aq3
kh[IAQ2] = k_aq2
kh[IAQ1] = k_aq1
kh[ICONF32] = 20.0 * LEAK_32               # 20 ft bed -> k33 giving the leakance
kh[ICONF21] = 20.0 * LEAK_21
# every AQL array carries a 0.5 multiplier in SHARP column 2, a landward half-cell
kh[ICONF32, 0, 0] *= 0.5
kh[ICONF21, 0, 0] *= 0.5
k33 = kh.copy()

por = 1.0e-4                               # POR; sets only the path to steady state

head_top = sharp_row2(INPUT, 292)[1:27]    # HEAD array, SHARP cols 2-27
bath = sharp_row2(INPUT, 302)[1:27]        # BATH: sea floor elevation
ONSHORE = int((bath == 0.0).sum())         # cols 2-6 are onshore
print(f"head_top onshore: {head_top[:ONSHORE]}")
print(f"head_top offshore (freshwater equivalent of 10 ft of seawater): "
      f"{head_top[ONSHORE]}")
print(f"check: -10 + 10*gams/gamf = {-10 + 10 * gams / gamf:.4f} ft")
print(f"sea floor offshore (BATH) = {bath[ONSHORE]} ft, shoreline at column "
      f"{ONSHORE + 1}/{ONSHORE + 2}")

## Boundaries

From `INPUT.T2`: a head-dependent boundary on top of the upper aquifer with conductance
$AQL\,\Delta x \Delta y$, whose head is the water table onshore and 0.25 ft offshore
(the freshwater-equivalent head of 10 ft of seawater on a -10 ft sea floor); a
saltwater GHB at head 0 offshore; constant saltwater head in SHARP column 27 in all
three aquifers; no flow elsewhere.

In [ ]:
def build_model(ws=sim_ws, perlen=6.0e10, nstp=120, tsmult=1.12):
    sim = flopy.mf6.MFSimulation(sim_name="capemay", sim_ws=ws, exe_name=mf6exe)
    flopy.mf6.ModflowTdis(sim, nper=1, time_units="seconds",
                          perioddata=[(perlen, nstp, tsmult)])
    ims = flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        no_ptcrecord=True,
        outer_maximum=500,
        inner_maximum=200,
        outer_dvclose=1.0e-5,          # feet: 1e-8 leaves the solver in a limit cycle
        inner_dvclose=1.0e-10,
        linear_acceleration="bicgstab",
        under_relaxation="DBD",
        under_relaxation_gamma=0.1,
        under_relaxation_theta=0.7,
        under_relaxation_kappa=0.07,
        under_relaxation_momentum=0.0,
        backtracking_number=20,
        backtracking_tolerance=1.05,
        backtracking_reduction_factor=0.1,
        backtracking_residual_limit=0.002,
    )

    hf0 = np.zeros((nlay, 1, ncol))
    hf0[:, 0, :] = np.maximum(head_top, 0.0)

    for is_saltwater in (False, True):
        name = "saltwater" if is_saltwater else "freshwater"
        gwf = flopy.mf6.ModflowGwf(sim, modelname=name, save_flows=True,
                                   newtonoptions="NEWTON")
        flopy.mf6.ModflowGwfdis(gwf, nlay=nlay, nrow=1, ncol=ncol, delr=delr,
                                delc=delc, top=top, botm=botm)
        flopy.mf6.ModflowGwfic(
            gwf, strt=np.zeros((nlay, 1, ncol)) if is_saltwater else hf0)
        flopy.mf6.ModflowGwfnpf(gwf, icelltype=0, k=kh, k33=k33,
                                save_specific_discharge=True,
                                save_saturation=True)
        flopy.mf6.ModflowGwfsto(gwf, iconvert=0, ss=0.0, sy=por,
                                transient={0: True})
        flopy.mf6.ModflowGwfswi(gwf, zeta_filerecord=f"{name}.zta")

        cond = LEAK_TOP * delr * delc
        cond[0] *= 0.5                      # AQL = 0.5 in SHARP column 2
        if is_saltwater:
            flopy.mf6.ModflowGwfghb(
                gwf, stress_period_data=[[IAQ3, 0, j, 0.0, cond[j]]
                                         for j in range(ONSHORE, ncol)])
            flopy.mf6.ModflowGwfchd(
                gwf, stress_period_data=[[k, 0, ncol - 1, 0.0]
                                         for k in (IAQ3, IAQ2, IAQ1)])
        else:
            flopy.mf6.ModflowGwfghb(
                gwf, stress_period_data=[[IAQ3, 0, j, head_top[j], cond[j]]
                                         for j in range(ncol)])
        flopy.mf6.ModflowGwfoc(gwf, head_filerecord=f"{name}.hds",
                               budget_filerecord=f"{name}.bud",
                               saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")])

    flopy.mf6.ModflowSwiswi(sim, exgtype="SWI6-SWI6",
                            exgmnamea="freshwater", exgmnameb="saltwater")
    sim.register_ims_package(ims, ["freshwater", "saltwater"])
    return sim


def run(ws=sim_ws, **kw):
    sim = build_model(ws, **kw)
    sim.write_simulation(silent=True)
    ok, buff = sim.run_simulation(silent=True)
    if not ok:
        print("\n".join(buff[-30:]))
        raise RuntimeError(f"{ws} did not converge")
    zo = flopy.utils.HeadFile(pl.Path(ws) / "freshwater.zta", text="zeta")
    return np.array(zo.times), zo.get_alldata().reshape(-1, nlay, ncol)

In [ ]:
t, z = run()
zeta = z[-1]
print(f"steady state reached: max |d zeta| over the last step = "
      f"{np.abs(z[-1] - z[-2]).max():.2e} ft")

## Comparison with SHARP

`RESULTS.T2` carries `ZINT` unclamped, so SHARP values outside an aquifer mean the cell
is entirely one fluid; SWI's zeta file returns the layer top or bottom in that case. The
two are compared only where SHARP's interface lies inside the aquifer.

In [ ]:
LAYERS = [("aquifer 1 (bottom)", "ZINT1", IAQ1, -190.0, -110.0),
          ("aquifer 2 (middle)", "ZINT2", IAQ2, -90.0, -50.0),
          ("aquifer 3 (top)", "ZINT3", IAQ3, -30.0, -10.0)]

for name, key, lay, lo, hi in LAYERS:
    v = sharp[key]
    inside = [c for c in range(2, 28) if lo + 1e-6 < v[c - 1] < hi - 1e-6]
    print(f"\n--- {name}: interface inside the aquifer in SHARP cols {inside}")
    if not inside:
        print("    SHARP has no interface in this aquifer anywhere")
        continue
    print(f"{'col':>5} {'x (ft)':>8} {'SHARP':>9} {'SWI':>9} {'diff':>8}")
    d = []
    for c in inside:
        j = c - 2
        print(f"{c:5d} {x[j]:8.0f} {v[c-1]:9.2f} {zeta[lay, j]:9.2f} "
              f"{zeta[lay, j] - v[c-1]:+8.2f}")
        d.append(zeta[lay, j] - v[c - 1])
    d = np.array(d)
    print(f"      mean {d.mean():+.2f} ft, RMSE {np.sqrt((d ** 2).mean()):.2f} ft")

## Freshwater heads

`RESULTS.T2` also carries `PHIF`. The saltwater is essentially static, so
$\zeta \approx -40\,h_f$ and a 0.3 ft head difference is a 12 ft interface difference.

In [ ]:
hf = flopy.utils.HeadFile(sim_ws / "freshwater.hds").get_alldata()[-1]

for key, lay, nm in [("PHIF1", IAQ1, "aquifer 1 (bottom)"),
                     ("PHIF2", IAQ2, "aquifer 2 (middle)"),
                     ("PHIF3", IAQ3, "aquifer 3 (top)")]:
    v = sharp[key]
    print(f"--- {nm}: freshwater head (ft), SHARP cols 2-12   [onshore = 2-6]")
    print("  col   " + "".join(f"{c:8d}" for c in range(2, 13)))
    print("  SHARP " + "".join(f"{v[c-1]:8.3f}" for c in range(2, 13)))
    print("  SWI   " + "".join(f"{hf[lay, 0, c-2]:8.3f}" for c in range(2, 13)))
    print("  diff  " + "".join(f"{hf[lay, 0, c-2] - v[c-1]:+8.3f}" for c in range(2, 13)))
    print()

print("does the head deficit explain the interface offset?  (x alphaf = 40)")
print(f"{'aquifer':>10} {'col':>4} {'dh_f':>8} {'dh_f x 40':>11} {'d zeta':>9}")
for hk, zk, lay, nm in [("PHIF1", "ZINT1", IAQ1, "1"), ("PHIF2", "ZINT2", IAQ2, "2")]:
    for c in (6, 9, 13, 14):
        j = c - 2
        lo, hi = (-190.0, -110.0) if lay == IAQ1 else (-90.0, -50.0)
        if not (lo + 1e-6 < sharp[zk][c-1] < hi - 1e-6):
            continue
        dh = hf[lay, 0, j] - sharp[hk][c-1]
        dz = zeta[lay, j] - sharp[zk][c-1]
        print(f"{nm:>10} {c:4d} {dh:+8.3f} {-alphaf * dh:+11.2f} {dz:+9.2f}")

## SWI on SHARP figure 19A

Figure 19A as background (extent -16000 to 68000 ft by -190 to -10 ft) with the SWI
interface in red. Solid lines are SHARP method 1 (complete mixing), dashed lines are
method 2 (restricted mixing), and the contours are Hill's SUTRA chlorides. SWI
corresponds to method 2. In the upper aquifer the SWI interface is the flat segment just
below -10 ft.

In [ ]:
import matplotlib.patheffects as pe

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(plt.imread("../data/sharp_fig19a.png"),
          extent=[-16000, 68000, -190, -10], aspect="auto", cmap="gray",
          interpolation="antialiased", zorder=0)

halo = [pe.Stroke(linewidth=3.6, foreground="white"), pe.Normal()]
for name, key, lay, lo, hi in LAYERS:
    zz = zeta[lay].copy()
    zz = np.ma.masked_where((zz <= lo + 1e-6) | (zz >= hi - 1e-6), zz)
    ax.plot(x, zz, "-", color="red", lw=2.2, zorder=3, path_effects=halo)

ax.plot([], [], "-", color="red", lw=2.2, label="SWI (this notebook)")
ax.plot([], [], "--", color="k", lw=1.4, label="SHARP, method 2 (restricted mixing)")
ax.plot([], [], "-", color="k", lw=1.4, label="SHARP, method 1 / SUTRA contours")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=9,
          frameon=False)
ax.set_xlim(-16000, 68000)
ax.set_ylim(-190, -10)
ax.set_xlabel("DISTANCE FROM SHORELINE, IN FEET")
ax.set_ylabel("ALTITUDE, IN FEET")
ax.set_title("Cape May cross-section: SWI on SHARP fig. 19A")
fig.tight_layout()

## Notes

- SWI interface elevations are 12-25 ft above SHARP's in aquifers 1 and 2. The offset is
  a freshwater head difference of 0.3-0.6 ft (SWI lower) times $\alpha_f$ = 40; the
  table above shows agreement to 0.05 ft.
- Onshore the heads agree to 0.004 ft. Offshore in the upper aquifer SHARP deactivates
  the freshwater equation (`ICODE = S`, `PHIF` near 0) whereas SWI keeps those cells at
  the equivalent freshwater head of 0.25 ft with zeta at the cell top.
- The controlling difference is that SHARP converts part of the freshwater leaking
  upward into saltwater (figure 24: 3.0e-3 $\mathrm{ft}^3$/s in, 2.0e-3 converted, 1.0e-3 to the
  sea). SWI conserves both fluids, so its throughflow of 1.75e-3 $\mathrm{ft}^3$/s leaves through
  a thin (0.14 ft) freshwater layer beneath the sea floor. Hill's SUTRA run also has
  freshwater offshore beneath the seabed, which SHARP's mixing rule removes.
- SHARP snaps zeta to a cell boundary when a zone is below 1 percent of cell thickness.
  Throttling SWI's offshore freshwater boundary to emulate this moves the interface the
  right way but needs a 300-fold cut to close half the gap, at which point the offshore
  upper aquifer is entirely freshwater; suppressing the boundary removes freshwater's
  only sink and the run fails. The threshold works in SHARP only because it is paired
  with the conversion.
- The saltwater GHB is applied offshore only. With the freshwater GHB at its equivalent
  freshwater head, both GHB fluxes vanish in a cell with no freshwater when $h_s$ equals
  the sea's saltwater head. Onshore, in a cell full of freshwater, a saltwater GHB would
  inject saltwater that is not there.

## Other notes

- Solver: `outer_dvclose` of 1e-5 ft (at 1e-8 the solver cycles with a residual of 8e-7
  and head changes of 1e-3 ft). DBD under-relaxation is needed in addition to
  backtracking; without it the run fails around time step 39 in the confining beds, and
  tightening `inner_dvclose` alone makes it fail earlier.
- Confining beds are explicit 20 ft layers (MF6 layers must be contiguous), so the model
  has 180 ft of SWI-active thickness against SHARP's 140 ft and the interface can occupy
  the beds. Their vertical conductance matches SHARP's leakance; the aquifer
  half-thickness terms are negligible (0.2 percent).
- The 0.5 multiplier on AQL in SHARP column 2 is reproduced; removing it changes the
  interface by 0.01 ft.
- Mass balance: 1.75e-3 $\mathrm{ft}^3$/s in onshore and out offshore, against SHARP's 3.0e-3 in
  and 1.0e-3 out; the difference is SHARP's conversion.